# SoundAQnet — CLI Tutorial

Every step of the pipeline is available as a standalone command-line tool.
This notebook shows the full CLI workflow with shell cells (`!` prefix).

| Command | Purpose |
|---|---|
| `soundaqnet-extract-mel` | Extract log-mel spectrograms → `.npy` |
| `soundaqnet-extract-loudness` | Extract ISO 532-1 loudness → `.npy` |
| `soundaqnet-infer` | Run inference → one complete CSV table |
| `soundaqnet-to-df` | Convert older legacy txt result folders → CSV tables |
| `soundaqnet-emosoundscape` | Predict EmoSoundscape valence/arousal from audio or `.npy` features |
| `soundaqnet-extract-emo-features` | Extract package-native 122-feature EmoSoundscape tables |

---

> **Setup** — commands are available after `pip install soundaqnet`.  
> Run cells with `Shift+Enter`. Replace the example paths with your own.

## 0 · Check installation


In [ ]:
!soundaqnet-extract-mel --help

In [ ]:
!soundaqnet-extract-loudness --help

In [ ]:
!soundaqnet-infer --help

In [ ]:
!soundaqnet-emosoundscape --help


In [ ]:
!soundaqnet-extract-emo-features --help


---
## 1 · Step 1 — Extract log-mel spectrograms

```
soundaqnet-extract-mel
    --input_dir   <directory of audio files>
    --output_dir  <directory for .npy outputs>
    --num_workers <parallel threads, default 4>
```

Supported formats: `.wav`, `.mp3`, `.flac`, `.ogg`, `.aiff`, `.m4a`, `.opus`  
Output: one `<stem>.npy` file per clip, shape `(frames, 64)`, dtype `float32`.


In [ ]:
AUDIO_DIR    = "audio/"            # ← replace with your audio directory
MEL_DIR      = "mel_features/"
LOUDNESS_DIR = "loudness_features/"

# Basic extraction (4 worker threads)
!soundaqnet-extract-mel \
    --input_dir  {AUDIO_DIR} \
    --output_dir {MEL_DIR} \
    --num_workers 4

In [ ]:
# Inspect the output
import numpy as np, pathlib

npy_files = sorted(pathlib.Path(MEL_DIR).glob("*.npy"))
print(f"Found {len(npy_files)} mel .npy files")
if npy_files:
    arr = np.load(npy_files[0])
    print(f"  {npy_files[0].name}: shape={arr.shape}, dtype={arr.dtype}")

---
## 2 · Step 2 — Extract ISO 532-1 loudness

```
soundaqnet-extract-loudness
    --input_dir   <directory of audio files>
    --output_dir  <directory for .npy outputs>
    --num_workers <parallel workers, default = cpu_count>
    --target_sr   <48000 | 44100 | 32000>  (default 48000)
    --method      <Varying | Stationary>   (Windows only; default Varying)
    --sound_field <Free | Diffuse>         (Windows only; default Free)
    --start_idx / --end_idx                (optional file-list slice)
    --overwrite                            (re-extract already-done files)
```

Output: one `<stem>.npy` file per clip, shape `(T, 1)`, dtype `float32` — total
time-varying loudness in **sone**.

> **Platform**: Windows uses the bundled `ISO_532-1.exe`; macOS/Linux uses mosqito and the bundled 1 kHz / 60 dB SPL calibration file.


In [ ]:
!soundaqnet-extract-loudness \
    --input_dir  {AUDIO_DIR} \
    --output_dir {LOUDNESS_DIR} \
    --num_workers 4 \
    --target_sr 48000 \
    --method Varying

In [ ]:
npy_loud = sorted(pathlib.Path(LOUDNESS_DIR).glob("*.npy"))
print(f"Found {len(npy_loud)} loudness .npy files")
if npy_loud:
    arr = np.load(npy_loud[0])
    print(f"  {npy_loud[0].name}: shape={arr.shape}, dtype={arr.dtype}")

### Verifying paired files

Each mel `.npy` must have a matching loudness `.npy` with the same stem.


In [ ]:
mel_stems  = {p.stem for p in pathlib.Path(MEL_DIR).glob("*.npy")}
loud_stems = {p.stem for p in pathlib.Path(LOUDNESS_DIR).glob("*.npy")}

missing_loud = mel_stems - loud_stems
missing_mel  = loud_stems - mel_stems

if missing_loud:
    print(f"WARNING: {len(missing_loud)} mel files have no matching loudness: {missing_loud}")
elif missing_mel:
    print(f"WARNING: {len(missing_mel)} loudness files have no matching mel: {missing_mel}")
else:
    print(f"All {len(mel_stems)} files are paired correctly.")

---
## 3 · Step 3 — Run inference

```
soundaqnet-infer
    --dataset_mel           <mel .npy directory>
    --dataset_wav_loudness  <loudness .npy directory>
    --output_csv            <complete results CSV>         (default: soundAQ.csv)
    --model                 <bundled name or /path/to.pth> (optional)
    --batch_size            <clips per forward pass>       (default 32)
    --device                <cpu | cuda | mps>             (optional)
    --resume                                               (skip existing legacy txt outputs)
    --legacy_txt                                           (also write old per-clip txt folders)
    --list-models                                          (show available bundled names)
```

The default output is one CSV containing scene, ISO, PAQ, top event labels,
full event ranking, and all 15 event probabilities. Use `--legacy_txt` only if
another workflow still depends on the historical per-clip txt folders.

In [ ]:
# List available bundled model names
!soundaqnet-infer --list-models

In [ ]:
# Run inference with the default model and write all results directly to CSV
!soundaqnet-infer \
    --dataset_mel          {MEL_DIR} \
    --dataset_wav_loudness {LOUDNESS_DIR} \
    --output_csv soundaqnet_predictions.csv


In [ ]:
# Use a specific model variant
!soundaqnet-infer \
    --dataset_mel          {MEL_DIR} \
    --dataset_wav_loudness {LOUDNESS_DIR} \
    --model SoundAQnet_ASC96_AEC95_PAQ1052 \
    --output_csv soundaqnet_predictions_v2.csv


In [ ]:
# Inspect the direct CSV output
import pandas as pd

results_df = pd.read_csv("soundaqnet_predictions.csv")
print(results_df.shape)
results_df.head()

---
## 4 · Optional legacy txt compatibility

The old two-step path is still available for projects that already depend on
these folders:

- `SoundAQnet_event_probability/`
- `SoundAQnet_scene_ISOPl_ISOEv_PAQ8DAQs/`

First run inference with `--legacy_txt`, then convert those txt files with
`soundaqnet-to-df`. New workflows should use the direct `--output_csv` file from
`soundaqnet-infer` instead.

In [ ]:
!soundaqnet-to-df --help

In [ ]:
# Optional: write legacy txt folders as well as the direct CSV
!soundaqnet-infer \
    --dataset_mel          {MEL_DIR} \
    --dataset_wav_loudness {LOUDNESS_DIR} \
    --output_csv soundaqnet_predictions.csv \
    --legacy_txt

# Convert legacy folders if you need the old split CSV outputs
!soundaqnet-to-df \
    --export both \
    --paq_dir SoundAQnet_scene_ISOPl_ISOEv_PAQ8DAQs \
    --event_dir SoundAQnet_event_probability \
    --output_prefix soundaqnet_legacy


In [ ]:
legacy_aq_df = pd.read_csv("soundaqnet_legacy.csv")
legacy_event_rank_df = pd.read_csv("soundaqnet_legacyEventRank.csv")

print(legacy_aq_df.shape, legacy_event_rank_df.shape)
legacy_aq_df.head()

---
## 5 · EmoSoundscape — valence/arousal CLI

`SoundAQnet` and `EmoSoundscape` are separate prediction paths. Use `soundaqnet-emosoundscape` when you want valence and arousal from the bundled `EmoS_gradient_boosting` model.

```
soundaqnet-emosoundscape
    --audio_file <clip.wav>                 (can be repeated)
    --audio_dir  <directory of audio files>
    --feature_dir <directory of .npy features>
    --model EmoS_gradient_boosting          (default)
    --output_csv <output CSV path>
```


In [ ]:
# List available bundled EmoSoundscape model names
!soundaqnet-emosoundscape --list-models


In [ ]:
# Single file or repeated explicit files
!soundaqnet-emosoundscape \
    --audio_file clip.wav \
    --output_csv emosoundscape_predictions.csv


In [ ]:
# Whole audio directory
!soundaqnet-emosoundscape \
    --audio_dir {AUDIO_DIR} \
    --output_csv emosoundscape_predictions.csv


In [ ]:
# Inspect the output
emo_df = pd.read_csv("emosoundscape_predictions.csv")
print(emo_df.shape)
emo_df.head()


### Optional: extract EmoSoundscape features first

This writes a CSV feature table with one row per clip. The direct `soundaqnet-emosoundscape --audio_dir ...` path is usually simpler for prediction.


In [ ]:
!soundaqnet-extract-emo-features \
    --audio_dir {AUDIO_DIR} \
    --output_csv emosoundscape_features.csv

emo_features = pd.read_csv("emosoundscape_features.csv")
print(emo_features.shape)
emo_features.head()


---
## 6 · SoundAQnet full pipeline in one shell script

Feature extraction plus direct CSV inference for scripted/batch use:

In [ ]:
%%bash
set -e

AUDIO_DIR="audio/"
MEL_DIR="mel_features/"
LOUD_DIR="loudness_features/"
WORKERS=4
OUTPUT_CSV="soundaqnet_predictions.csv"

echo "=== Step 1: Extract mel spectrograms ==="
soundaqnet-extract-mel \
    --input_dir  "$AUDIO_DIR" \
    --output_dir "$MEL_DIR" \
    --num_workers "$WORKERS"

echo "=== Step 2: Extract loudness ==="
soundaqnet-extract-loudness \
    --input_dir  "$AUDIO_DIR" \
    --output_dir "$LOUD_DIR" \
    --num_workers "$WORKERS"

echo "=== Step 3: Run inference and write complete CSV ==="
soundaqnet-infer \
    --dataset_mel          "$MEL_DIR" \
    --dataset_wav_loudness "$LOUD_DIR" \
    --batch_size 32 \
    --output_csv "$OUTPUT_CSV"

echo "Done. Results in ${OUTPUT_CSV}"


---
## 7 · CLI vs Python API — when to use which

| Situation | Recommendation |
|---|---|
| Large dataset, scripted pipeline | **CLI** — writes one complete CSV directly and is easy to log |
| Interactive analysis, Jupyter | **Python API** — returns DataFrames directly |
| Single file or small batch | **Python API** `predict_from_audio` — fewest steps |
| Integrating into another codebase | **Python API** — importable and typed |
| HPC / cluster job | **CLI** — use `--start_idx` / `--end_idx` for loudness slices and `--resume` if using legacy txt outputs |

The Python `SoundAQnet.predict(...)` DataFrame and CLI `soundaqnet-infer --output_csv` table come from the same model predictions. `soundaqnet-to-df` remains available for converting older legacy txt folders. `EmoSoundscape.predict_from_audio(...)` and `soundaqnet-emosoundscape` likewise use the same valence/arousal model.